# Digital Twin for IPA — quickstart

A machine-readable twin of India's **Investment Promotion Apparatus**: 312 incentive instruments (86 central entities + 30 states/UTs), a source-verified scheme-interlinkage graph, PRS budget overlay, and live portal health checks.

Run all cells (Runtime → Run all). Takes ~2 minutes; the only network calls are `git clone` and the live portal probes in the last section.


In [ ]:
# 1. Clone the twin
!git clone -q --depth 1 https://github.com/herrrickshaw/digital-twin-for-ipa.git
%cd digital-twin-for-ipa
!ls layers/


In [ ]:
# 2. Load the flat instrument index -- every instrument in one schema
import json, pandas as pd
idx = json.load(open('layers/13_flat_instrument_index.json'))
df = pd.DataFrame(idx['instruments'])
print(f"{idx['count']} instruments | by tier: {df.tier.value_counts().to_dict()}")
df[['instrument','offering_entity','tier','instrument_type']].sample(10, random_state=7)


In [ ]:
# 3. Who offers the most instruments?
import matplotlib.pyplot as plt
top = df.offering_entity.value_counts().head(15)[::-1]
top.plot.barh(figsize=(8,5), title='Top 15 offering entities by instrument count')
plt.tight_layout(); plt.show()


In [ ]:
# 4. What's OPEN right now? (grep the status/application fields)
mask = df[['status','application_status','what_companies_get']].fillna('').agg(' '.join, axis=1).str.lower()
open_now = df[mask.str.contains('open|live|active|in force|operative')]
closed = df[mask.str.contains('oversubscribed|lapsed|superseded')]
print(f'open/active-flagged: {len(open_now)} | closed/lapsed-flagged: {len(closed)}')
open_now[['instrument','offering_entity','tier']].head(15)


In [ ]:
# 5. The interlinkage graph -- how schemes stack, exclude, feed and converge
g = json.load(open('layers/10_interlinkages.json'))
edges = pd.DataFrame([{'schemes':' + '.join(e['schemes']), 'rel':e['relationship'], 'src':e['source_type']} for e in g['interlinkages']])
print(edges.rel.value_counts().to_dict())
print('\nVerification tally (vs Drishti + 6 IAS sources):')
print(json.dumps(g['verification_2026_07_20']['tally'], indent=1))
edges


In [ ]:
# 6. Budget reality check (PRS overlay) -- announced vs utilized
prs = json.load(open('layers/11_prs_budget_layer.json'))
for m in prs['ministry_findings'][:4]:
    print(f"== {m['ministry']} ({m['budget_year']})")
    for f in m['findings'][:2]: print('  -', f['figure'], '|', f['fact'][:110])


In [ ]:
# 7. LIVE: probe a few portals + the UNNATI notice (network)
import subprocess
for name,url in [('NSWS','https://www.nsws.gov.in'), ('UNNATI','https://unnati.dpiit.gov.in/'), ('Gujarat IFP','https://ifp.gujarat.gov.in/DIGIGOV/'), ('TN single window','https://tnswp.com/DIGIGOV/swp-tnswp.jsp')]:
    code_ = subprocess.run(['curl','-sL','-o','/dev/null','-w','%{http_code}','-m','20','-A','Mozilla/5.0',url], capture_output=True, text=True).stdout
    print(f'{name:18s} HTTP {code_}  {url}')
body = subprocess.run(['curl','-sL','-m','30','-A','Mozilla/5.0','https://unnati.dpiit.gov.in/'], capture_output=True, text=True).stdout
print('\nUNNATI registration stopped (oversubscribed)?', 'has presently been stopped' in body)


In [ ]:
# 8. Regenerate the human-readable catalogue from the index
!python3 scripts/build_catalogue.py
print(open('docs/SCHEME_CATALOGUE.md').read()[:1500])


## Where to go next
- [`docs/SCHEME_CATALOGUE.md`](https://github.com/herrrickshaw/digital-twin-for-ipa/blob/main/docs/SCHEME_CATALOGUE.md) — all 312 instruments, grouped and status-flagged
- [`docs/DATA_MODEL.md`](https://github.com/herrrickshaw/digital-twin-for-ipa/blob/main/docs/DATA_MODEL.md) — the sources → model → products flowchart
- [`docs/DIRECTORY.md`](https://github.com/herrrickshaw/digital-twin-for-ipa/blob/main/docs/DIRECTORY.md) — 64 verified portals + 12 known-bad domains
- Companion repos: [india-trade-sector-policy-recommendations](https://github.com/herrrickshaw/india-trade-sector-policy-recommendations) · [focus-sector-investor-map](https://github.com/herrrickshaw/focus-sector-investor-map)
